# Table 2: comparison of downward-continuation approaches on synthetic data

Run notebooks `01`–`10` first (or `scripts/run_all.sh`). Each writes its rows to `../results/*.json`.
This notebook collects them into Table 2, checks every row against the reference values in
`gravdc/table2.py`, and writes `../results/table2.csv` and `../results/table2.md`.

**Evaluation protocol (all notebooks).** The 10 synthetic test grids (32 × 32 nodes, 50 m spacing) are
scaled to [-1, 1] per grid. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 grids. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import csv
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from gravdc.table2 import METRICS, ORDER, TABLE2, matches

RESULTS_DIR = "../results"
runs = {}
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith(".json"):
        r = json.load(open(os.path.join(RESULTS_DIR, f)))
        runs[r["method"]] = r
missing = [n for n in ORDER if n not in runs]
assert not missing, f"run notebooks 01-10 first; missing rows: {missing}"
assert all(len(runs[n]["per_image"]) == 10 for n in ORDER), "every row must be averaged over the 10 test grids"

In [2]:
def fmt(v):
    return f"{v[0]:.4f} ± {v[1]:.4f}"

print(f"{'Approach':34s}" + "".join(f"{k:>20s}" for k in METRICS) + "   same as Table 2")
print("-" * 134)
n_same = 0
for name in ORDER:
    s = runs[name]["summary"]
    same = matches(s, name) if name in TABLE2 else None
    n_same += bool(same)
    flag = {True: "✓", False: "✗", None: "-"}[same]
    print(f"{name:34s}" + "".join(f"{fmt(s[k]):>20s}" for k in METRICS) + f"   {flag}")
print(f"\n{n_same}/{len(ORDER)} rows identical to Table 2 (all four metrics, mean and std, 4 decimals)")

Approach                                          PSNR                SSIM                  RE                RMSE   same as Table 2
--------------------------------------------------------------------------------------------------------------------------------------
TTSIDC                                47.9276 ± 4.5850     0.9985 ± 0.0014     0.0135 ± 0.0077     0.0094 ± 0.0067   ✓
Oracle TRDC                           37.0393 ± 1.9052     0.9761 ± 0.0082     0.0424 ± 0.0040     0.0282 ± 0.0066   ✓
TRDC                                  36.3714 ± 1.7217     0.9756 ± 0.0057     0.0457 ± 0.0039     0.0303 ± 0.0064   ✓
DU-Net                                48.6022 ± 2.0723     0.9984 ± 0.0007     0.0113 ± 0.0018     0.0075 ± 0.0019   ✓
DC-Net                                52.3368 ± 3.9071     0.9995 ± 0.0005     0.0078 ± 0.0034     0.0053 ± 0.0029   ✓
U-Net                                 57.8891 ± 3.3056     0.9999 ± 0.0001     0.0040 ± 0.0015     0.0027 ± 0.0013   ✓
Physics-Trained   

## Write `results/table2.csv` and `results/table2.md`
Bold: best value of each column.

In [3]:
# best row of each column (at full precision, so ties at 4 decimals are broken)
best = {k: (max if k in ("PSNR", "SSIM") else min)(ORDER, key=lambda n: runs[n]["summary"][k][0]) for k in METRICS}

with open(os.path.join(RESULTS_DIR, "table2.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Approach"] + [f"{k}_{stat}" for k in METRICS for stat in ("mean", "std")])
    for name in ORDER:
        w.writerow([name] + [f"{v:.4f}" for k in METRICS for v in runs[name]["summary"][k]])

lines = ["| Approach | PSNR (dB) ↑ | SSIM ↑ | RE ↓ | RMSE ↓ |", "|---|---|---|---|---|"]
for name in ORDER:
    s = runs[name]["summary"]
    cells = [f"**{fmt(s[k])}**" if name == best[k] else fmt(s[k]) for k in METRICS]
    lines.append(f"| {name} | " + " | ".join(cells) + " |")
open(os.path.join(RESULTS_DIR, "table2.md"), "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

| Approach | PSNR (dB) ↑ | SSIM ↑ | RE ↓ | RMSE ↓ |
|---|---|---|---|---|
| TTSIDC | 47.9276 ± 4.5850 | 0.9985 ± 0.0014 | 0.0135 ± 0.0077 | 0.0094 ± 0.0067 |
| Oracle TRDC | 37.0393 ± 1.9052 | 0.9761 ± 0.0082 | 0.0424 ± 0.0040 | 0.0282 ± 0.0066 |
| TRDC | 36.3714 ± 1.7217 | 0.9756 ± 0.0057 | 0.0457 ± 0.0039 | 0.0303 ± 0.0064 |
| DU-Net | 48.6022 ± 2.0723 | 0.9984 ± 0.0007 | 0.0113 ± 0.0018 | 0.0075 ± 0.0019 |
| DC-Net | 52.3368 ± 3.9071 | 0.9995 ± 0.0005 | 0.0078 ± 0.0034 | 0.0053 ± 0.0029 |
| U-Net | 57.8891 ± 3.3056 | 0.9999 ± 0.0001 | 0.0040 ± 0.0015 | 0.0027 ± 0.0013 |
| Physics-Trained | 26.9830 ± 1.0557 | 0.8539 ± 0.0810 | 0.1378 ± 0.0346 | 0.0878 ± 0.0101 |
| CNN-regularized | 50.7438 ± 8.5486 | 0.9982 ± 0.0040 | 0.0140 ± 0.0178 | 0.0102 ± 0.0140 |
| Denoising diffusion (U-Net) | 59.0080 ± 1.6162 | 0.9999 ± 0.0000 | 0.0034 ± 0.0004 | 0.0022 ± 0.0005 |
| Multiscale DiT (Single-shot) | 60.1563 ± 4.4481 | 0.9998 ± 0.0003 | 0.0033 ± 0.0020 | 0.0023 ± 0.0017 |
| Multiscale DiT (Stabi